In [1]:
# imports
import os
import numpy as np
import tensorflow as tf
from google.cloud import storage
import tempfile
# import cv2
# from drought_detection.params import BUCKET_NAME, BUCKET_TRAIN_DATA_PATH


In [ ]:
# # Marie's imports (suggested for later plotting of NDVI)
# import xarray as xr
# import holoviews as hv
# # import geoviews as gv
# import datashader as ds
# # import cartopy.crs as ccrs
# from holoviews import opts
# from holoviews.operation.datashader import rasterize, shade

## AVAILABLE BANDWIDTHS

In [76]:
# list of spectral band labels
bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11']

    # features = {
    #     'B1': tf.compat.v1.FixedLenFeature([], tf.string),    # 0.43 - 0.45 μm Coastal aerosol
    #     'B2': tf.compat.v1.FixedLenFeature([], tf.string),    # Blue
    #     'B3': tf.compat.v1.FixedLenFeature([], tf.string),    # Green
    #     'B4': tf.compat.v1.FixedLenFeature([], tf.string),    # Red
    #     'B5': tf.compat.v1.FixedLenFeature([], tf.string),    # Near infrared
    #     'B6': tf.compat.v1.FixedLenFeature([], tf.string),    # Shortwave infrared 1
    #     'B7': tf.compat.v1.FixedLenFeature([], tf.string),    # Shortwave infrared 2
    #     'B8': tf.compat.v1.FixedLenFeature([], tf.string),
    #     'B9': tf.compat.v1.FixedLenFeature([], tf.string),
    #     'B10': tf.compat.v1.FixedLenFeature([], tf.string),
    #     'B11': tf.compat.v1.FixedLenFeature([], tf.string),
    #     'label': tf.compat.v1.FixedLenFeature([], tf.int64),
    #     }

# Function: Retrieve and filter raw satellite data for specific spectral bands

In [6]:
# function to filter raw data for selected bands
def parse_visual(data):
    '''
    This function filters satellite image data by specific spectral bands (RGB in this case).
    The function loads a batch of satellite images from a list of files
    and parses the satellite image data files for some specific features,
    e.g. spectral bands (B2, B3, B4, see official documentation)

    Input(s): - list of satellite image files (including path, e.g '/data/train/part-r-00000')
    Outputs:  - list of dictionaries of raw satellite data (filtered by spectral band)
    '''
    dataset = tf.data.TFRecordDataset(data)
    # pattern for one part file
    # dataset = tf.data.TFRecordDataset('part-r-00099')
    iterator = tf.compat.v1.data.make_one_shot_iterator(dataset)

    features = {
        'B4': tf.compat.v1.FixedLenFeature([], tf.string), # red
        'B5': tf.compat.v1.FixedLenFeature([], tf.string), # infrared
        'label': tf.compat.v1.FixedLenFeature([], tf.int64),
    }

    parsed_examples = [tf.compat.v1.parse_single_example(data, features) for data in iterator]
    return parsed_examples


# Transform each raw satellite image into an RGB array with label & scale it (by 255)

In [25]:
# transform & scaling function

def get_img_from_example(parsed_example, intensify=True):
    '''
    This function creates an RGB 3D array in shape 65x65x3 (65x65 pixels) for 
    a single parsed satellite image, while also scaling each spectral band. 
    For each band (depends on filtering done by above function),
    the raw band data is transformed from the Tensorflow specific data format into
    a 2D array of dimension 65x65 pixels.
    
    Next, it does some scaling: if intensity=True, it divides each pixel by the
    maximum value of the pixels, and then multiplies it by 255.
    Otherwise, if intensify=False, it just multiplies each pixel in the matrix by 255.
    "When the range of pixel brightness values is closer to 0, a darker image is rendered by default.
    You can stretch the values to extend to the full 0-255 range of potential values to 
    increase the visual contrast of the image."
    - https://www.earthdatascience.org/courses/use-data-open-source-python/multispectral-remote-sensing/landsat-in-Python/
    
    Lastly, the function adds the corresponding label, and returns the image 2D array, as well as the label.
    

    Input: - a parsed satellite image: Specific Tensorflow format (as dictionary)
    Output(s) - satellite image & its label:
                - rgbArray: tuple of 3D numpy array (shape 65x65x3)
                - label: int32
    '''
    rgbArray = np.zeros((65,65,1), 'uint8')
    # for i, band in enumerate(['B4', 'B5']): # spectral bands
    band_data = np.frombuffer(parsed_example['B4'].numpy(), dtype=np.uint8) # convert raw sat data into 1D array of integers
    band_data = band_data.reshape(65, 65) # reshape image into matrix of 65x65 pixels
    if intensify:
        band_data = band_data/np.max(band_data)*255 # are we transforming the image from bytes to digital numbers? 
    else:
        band_data = band_data*255
    rgbArray[..., 0] = band_data
        
    label = tf.cast(parsed_example['label'], tf.int32).numpy() # get label
        
    return rgbArray, label

In [35]:
# transform & scaling function

def get_img_from_example_b5(parsed_example, intensify=True):
    '''
    This function creates an RGB 3D array in shape 65x65x3 (65x65 pixels) for 
    a single parsed satellite image, while also scaling each spectral band. 
    For each band (depends on filtering done by above function),
    the raw band data is transformed from the Tensorflow specific data format into
    a 2D array of dimension 65x65 pixels.
    
    Next, it does some scaling: if intensity=True, it divides each pixel by the
    maximum value of the pixels, and then multiplies it by 255.
    Otherwise, if intensify=False, it just multiplies each pixel in the matrix by 255.
    "When the range of pixel brightness values is closer to 0, a darker image is rendered by default.
    You can stretch the values to extend to the full 0-255 range of potential values to 
    increase the visual contrast of the image."
    - https://www.earthdatascience.org/courses/use-data-open-source-python/multispectral-remote-sensing/landsat-in-Python/
    
    Lastly, the function adds the corresponding label, and returns the image 2D array, as well as the label.
    

    Input: - a parsed satellite image: Specific Tensorflow format (as dictionary)
    Output(s) - satellite image & its label:
                - rgbArray: tuple of 3D numpy array (shape 65x65x3)
                - label: int32
    '''
    rgbArray = np.zeros((65,65,1), 'uint8')
    # for i, band in enumerate(['B4', 'B5']): # spectral bands
    band_data = np.frombuffer(parsed_example['B5'].numpy(), dtype=np.uint8) # convert raw sat data into 1D array of integers
    band_data = band_data.reshape(65, 65) # reshape image into matrix of 65x65 pixels
    if intensify:
        band_data = band_data/np.max(band_data)*255 # are we transforming the image from bytes to digital numbers? 
    else:
        band_data = band_data*255
    rgbArray[..., 0] = band_data
        
    label = tf.cast(parsed_example['label'], tf.int32).numpy() # get label
        
    return rgbArray, label

In [26]:
# create a function called dirlist that extracts a list of file names (file) from a directory (di)
dirlist = lambda di: [os.path.join(di, file) for file in os.listdir(di) if 'part-' in file]

# get list of files
validation_files = dirlist('../raw_data/val/')

# parse first file (returns image as list of dict)
parsed_examples = parse_visual(validation_files[0])

# parsed_examples[image][band]
parsed_examples[0]['B4']

<tf.Tensor: shape=(), dtype=string, numpy=b'511.0/.022/(()02..-01465467541344411.11457671-311310114356645564.421011/01350$\x1c)62,10..21342003533542./2354340-33120/014565546445-/11132101453- \x1f+9810./0232,,176345430-,/255622334300124543445213./.,./23134630)&),.1/022311//4864553323.+1577/24643125555563520,.,1/00./3335642240#\x1d,2332122234566752011-\',167.1354532443476674-++)220/0./127522450( #432345424214:75311..*(/44/3453443453569786/,/*3411/1--0455433,(\')432346512113;:7522.00*+121664/02234476977521.+232223-,-046654)$#5732235400//36:73230./,*/-27430/0114798:;6664/+2222321/-./3544+$ 8311244210/033653121/..**)244330/2378978;::940*13323111....1000*&6).0231////1334342101/.--+0221000/06899::::941*2321111243.04755-./ *//11--,13242-122030./10.2110.,-25::;:9:;832,2321111243/22433.24))-.//0//33241+,13110-.01.0120-,08;<9:78:<940-3210/34334/22433.24))-.//0233311/.,.300-+.10,-20//148;9795/08<6.*33342352343430/14.6<.*0/.1233311/.,.300-+012.-/-.014887770)\'3;41,43344321122201211/,+,)+./03011310+

In [51]:
image_b4, b4_label = get_img_from_example(parsed_examples[0])

In [52]:
image_b5, b5_label = get_img_from_example_b5(parsed_examples[0])

In [53]:
image_b4.shape

(65, 65, 1)

In [54]:
image_b4[0]

array([[211],
       [195],
       [195],
       [183],
       [191],
       [187],
       [183],
       [191],
       [199],
       [199],
       [187],
       [159],
       [159],
       [163],
       [191],
       [199],
       [183],
       [183],
       [179],
       [191],
       [195],
       [207],
       [215],
       [211],
       [207],
       [215],
       [219],
       [211],
       [207],
       [195],
       [203],
       [207],
       [207],
       [207],
       [195],
       [195],
       [183],
       [195],
       [195],
       [207],
       [211],
       [219],
       [215],
       [219],
       [195],
       [179],
       [203],
       [195],
       [195],
       [203],
       [195],
       [191],
       [195],
       [195],
       [207],
       [203],
       [211],
       [215],
       [215],
       [207],
       [211],
       [211],
       [215],
       [207],
       [183]], dtype=uint8)

In [108]:
image_b5[0][0][0].shape

()

In [56]:
image_b5

array([[[218],
        [209],
        [206],
        ...,
        [233],
        [230],
        [215]],

       [[218],
        [212],
        [206],
        ...,
        [230],
        [230],
        [212]],

       [[209],
        [209],
        [203],
        ...,
        [221],
        [224],
        [212]],

       ...,

       [[209],
        [209],
        [215],
        ...,
        [191],
        [160],
        [170]],

       [[206],
        [209],
        [212],
        ...,
        [203],
        [197],
        [206]],

       [[203],
        [209],
        [212],
        ...,
        [221],
        [215],
        [209]]], dtype=uint8)

In [78]:
x1 = np.squeeze(image_b4)
x2 = np.squeeze(image_b5)

In [79]:
x1

array([[211, 195, 195, ..., 215, 207, 183],
       [207, 199, 195, ..., 207, 211, 179],
       [187, 195, 195, ..., 195, 203, 183],
       ...,
       [199, 199, 203, ..., 167, 143, 139],
       [195, 195, 199, ..., 167, 163, 167],
       [195, 199, 199, ..., 191, 187, 183]], dtype=uint8)

In [80]:
x2

array([[218, 209, 206, ..., 233, 230, 215],
       [218, 212, 206, ..., 230, 230, 212],
       [209, 209, 203, ..., 221, 224, 212],
       ...,
       [209, 209, 215, ..., 191, 160, 170],
       [206, 209, 212, ..., 203, 197, 206],
       [203, 209, 212, ..., 221, 215, 209]], dtype=uint8)

In [105]:
x2[:][0]

array([218, 209, 206, 197, 200, 200, 197, 206, 215, 215, 212, 206, 221,
       227, 224, 215, 206, 203, 197, 203, 209, 221, 230, 230, 230, 233,
       230, 227, 224, 212, 221, 218, 221, 218, 209, 209, 203, 206, 206,
       212, 227, 230, 227, 230, 218, 206, 215, 215, 218, 221, 209, 212,
       215, 215, 224, 221, 224, 230, 233, 227, 230, 227, 233, 230, 215],
      dtype=uint8)

In [81]:
x2-x1

array([[ 7, 14, 11, ..., 18, 23, 32],
       [11, 13, 11, ..., 23, 19, 33],
       [22, 14,  8, ..., 26, 21, 29],
       ...,
       [10, 10, 12, ..., 24, 17, 31],
       [11, 14, 13, ..., 36, 34, 39],
       [ 8, 10, 13, ..., 30, 28, 26]], dtype=uint8)

In [101]:
m1 = np.array([[3, 4], 
               [5, 6],
               [3, 5]], dtype='uint8')

m2 = np.array([[8, 9], 
               [0, 6],
               [7, 4]], dtype='uint8')
r = np.add(m1, m2)
print(r)

[[11 13]
 [ 5 12]
 [10  9]]


In [94]:
np.array(x2[0])


array([218, 209, 206, 197, 200, 200, 197, 206, 215, 215, 212, 206, 221,
       227, 224, 215, 206, 203, 197, 203, 209, 221, 230, 230, 230, 233,
       230, 227, 224, 212, 221, 218, 221, 218, 209, 209, 203, 206, 206,
       212, 227, 230, 227, 230, 218, 206, 215, 215, 218, 221, 209, 212,
       215, 215, 224, 221, 224, 230, 233, 227, 230, 227, 233, 230, 215],
      dtype=uint8)

In [95]:

m1 = np.array([[3, 4], [5, 6]])
m2 = np.array([[8, 9], [0, 6]])
r = np.subtract(m1, m2)
print(r)

numpy.ndarray

In [71]:
ndvi = (x2 - x1) / (x2 + x1)
ndvi

array([[[0.04046243],
        [0.09459459],
        [0.07586207],
        [0.11290323],
        [0.06666667],
        [0.09923664],
        [0.11290323],
        [0.10638298],
        [0.10126582],
        [0.10126582],
        [0.17482517],
        [0.43119266],
        [0.5       ],
        [0.47761194],
        [0.20754717],
        [0.10126582],
        [0.17293233],
        [0.15384615],
        [0.15      ],
        [0.08695652],
        [0.09459459],
        [0.08139535],
        [0.07936508],
        [0.1027027 ],
        [0.12707182],
        [0.09375   ],
        [0.05699482],
        [0.08791209],
        [0.09714286],
        [0.11258278],
        [0.10714286],
        [0.06508876],
        [0.08139535],
        [0.06508876],
        [0.09459459],
        [0.09459459],
        [0.15384615],
        [0.07586207],
        [0.07586207],
        [0.03067485],
        [0.08791209],
        [0.05699482],
        [0.06451613],
        [0.05699482],
        [0.14649682],
        [0

In [72]:
ndvi.shape

(2, 65, 1)

In [63]:
ndvi[0].max()

0.5

In [75]:
image_b5[0][0], image_b4[0][0], ndvi[0][0]

(array([218], dtype=uint8), array([211], dtype=uint8), array([0.04046243]))

In [73]:
(image_b5[0:2] - image_b4[0:2])

array([[[ 7],
        [14],
        [11],
        [14],
        [ 9],
        [13],
        [14],
        [15],
        [16],
        [16],
        [25],
        [47],
        [62],
        [64],
        [33],
        [16],
        [23],
        [20],
        [18],
        [12],
        [14],
        [14],
        [15],
        [19],
        [23],
        [18],
        [11],
        [16],
        [17],
        [17],
        [18],
        [11],
        [14],
        [11],
        [14],
        [14],
        [20],
        [11],
        [11],
        [ 5],
        [16],
        [11],
        [12],
        [11],
        [23],
        [27],
        [12],
        [20],
        [23],
        [18],
        [14],
        [21],
        [20],
        [20],
        [17],
        [18],
        [13],
        [15],
        [18],
        [20],
        [19],
        [16],
        [18],
        [23],
        [32]],

       [[11],
        [13],
        [11],
        [12],
        [11],
        [11],
    

In [74]:
(image_b5[0:2] + image_b4[0:2])

array([[[173],
        [148],
        [145],
        [124],
        [135],
        [131],
        [124],
        [141],
        [158],
        [158],
        [143],
        [109],
        [124],
        [134],
        [159],
        [158],
        [133],
        [130],
        [120],
        [138],
        [148],
        [172],
        [189],
        [185],
        [181],
        [192],
        [193],
        [182],
        [175],
        [151],
        [168],
        [169],
        [172],
        [169],
        [148],
        [148],
        [130],
        [145],
        [145],
        [163],
        [182],
        [193],
        [186],
        [193],
        [157],
        [129],
        [162],
        [154],
        [157],
        [168],
        [148],
        [147],
        [154],
        [154],
        [175],
        [168],
        [179],
        [189],
        [192],
        [178],
        [185],
        [182],
        [192],
        [181],
        [142]],

       [[169],
        

In [110]:
import matplotlib.pyplot as plt

# single image
fig=plt.figure(figsize=(5, 10), dpi= 80, facecolor='w', edgecolor='k')

# image has been preprocessed (image is from blobs in cloud)
img = ndvi

plt.imshow(img[0:1]).axes.get_xaxis().set_visible(False)
plt.imshow(img[0:1]).axes.get_yaxis().set_visible(False)
# plt.title(str(label))

# we only need B4 (red) and B5 (NIR) for NDVI

## formula = (NIR - RED) / (NIR + RED)

ndvi = (band_nir.astype(float) - band_red.astype(float)) / (band_nir + band_red)

In [ ]:
def _parse_(serialized_example, keylist=['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8']):
example = tf.io.parse_single_example(serialized_example, features)

def getband(example_key):
    img = tf.compat.v1.decode_raw(example_key, tf.uint8)
    return tf.reshape(img[:IMG_DIM**2], shape=(IMG_DIM, IMG_DIM, 1))

bandlist = [getband(example[key]) for key in keylist]

# combine bands into tensor
image = tf.concat(bandlist, -1)

## format

output is in format: images[satellie image number][0 = 3D image matrix / 1 = label] 

the first index refers to the satellite image number (length depends on how many images you chose to)

# compute NDVI

https://medium.com/analytics-vidhya/python-for-geosciences-satellite-image-analysis-step-by-step-6d49b1ad567

In [ ]:
def normalized_difference(img, b1, b2, eps=0.0001):
    band1 = np.where((img[b1]==0) & (img[b2]==0), np.nan, img[b1])
    band2 = np.where((img[b1]==0) & (img[b2]==0), np.nan, img[b2])
    
    return (band1 - band2) / (band1 + band2)

# Calculating two indices
ndvi = normalized_difference(img, 'B5', 'B4')
mndwi = normalized_difference(img, 'B3', 'B5')

# checking the images
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(ndvi, cmap='Greens')
ax[1].imshow(mndwi, cmap='Blues')